In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
df_info = spark.table("bronze.tb_movies_info")

display(df_info)

In [0]:
from pyspark.sql.functions import (
    col,
    to_date,
    year,
    row_number,
    when,
    trim
)
from pyspark.sql.window import Window

In [0]:
from pyspark.sql.functions import regexp_extract, coalesce, try_to_date


df_info_tratada = df_info.withColumn(
    "data_lancamento",
    coalesce(
        try_to_date(
            trim(col("release_date")),
            "yyyy-MM-dd"
        ),
        try_to_date(
            trim(col("release_date")),
            "MM-dd-yyyy"
        ),
        try_to_date(
            trim(col("release_date")),
            "dd/MM/yyyy"
        ),
        try_to_date(
            trim(col("release_date")),
            "yyyy/MM/dd"
        )
    )
)


In [0]:
display(
    df_info_tratada
    .filter(col("release_date").isNotNull() & col("data_lancamento").isNull())
    .select("id", "title", "release_date")
)

In [0]:
display(
    df_info_tratada
    .filter(
        col("release_date").isNotNull() &
        col("data_lancamento").isNull()
    )
    .select("id", "title", "release_date")
)

In [0]:
from pyspark.sql.functions import count, when

df_info_tratada.select(
    count("*").alias("total_linhas"),
    count(
        when(col("data_lancamento").isNull(), 1)
    ).alias("datas_null")
).show()

In [0]:
display(
    df_info_tratada
    .groupBy("status")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
from pyspark.sql.functions import lower, regexp_replace


df_info_tratada = df_info_tratada.withColumn(
    "status_normalizado",
    when(
        lower(trim(col("status"))) == "released",
        "Lançado"
    )
    .when(
        regexp_replace(lower(trim(col("status"))), "-", " ") == "post production",
        "Pós-Produção"
    )
    .when(
        regexp_replace(lower(trim(col("status"))), "-", " ") == "in production",
        "Em Produção"
    )
    .when(
        lower(trim(col("status"))) == "planned",
        "Planejado"
    )
    .when(
        lower(trim(col("status"))) == "rumored",
        "Rumores"
    )
    .when(
        lower(trim(col("status"))) == "canceled",
        "Cancelado"
    )
    .otherwise("Não Informado")
)


display(
    df_info_tratada
    .groupBy("status_normalizado")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
df_info_tratada \
    .groupBy("id") \
    .count() \
    .filter(col("count") > 1) \
    .count()

In [0]:
window_id = Window.partitionBy("id").orderBy(
    col("ingestion_datetime").desc()
)

df_info_deduplicada = (
    df_info_tratada
    .withColumn("rn", row_number().over(window_id))
    .filter(col("rn") == 1)
    .drop("rn")
)

display(df_info_deduplicada)

In [0]:
print("Antes:", df_info_tratada.count())
print("Depois:", df_info_deduplicada.count())

In [0]:
df_info_final = df_info_deduplicada.withColumn(
    "ano_lancamento",
    year(col("data_lancamento"))
)

display(
    df_info_final.select(
        "id",
        "title",
        "data_lancamento",
        "ano_lancamento",
        "status_normalizado"
    )
)

In [0]:
df_info_final = df_info_final.select(
    col("id").alias("id_filme"),
    col("tconst").alias("id_imdb"),
    col("title").alias("titulo"),
    col("original_title").alias("titulo_original"),
    col("original_language").alias("idioma_original"),
    col("data_lancamento"),
    col("ano_lancamento"),
    col("runtime").alias("duracao"),
    col("status_normalizado").alias("status"),
    col("overview").alias("sinopse"),
    col("tagline").alias("slogan"),
    col("ingestion_datetime")
)

display(df_info_final)

In [0]:
df_info_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.tb_info_filmes")

display(spark.table("silver.tb_info_filmes").limit(10))

In [0]:
df_financeiro = spark.table("bronze.tb_movies_financials")

display(df_financeiro.limit(10))

In [0]:

from pyspark.sql.functions import (
    col, trim, regexp_replace, expr, when, lit, round, upper
)

# 1. Ler os dados financeiros da Bronze
df_financeiro = spark.table("bronze.tb_movies_financials")


# 2. Limpar orçamento e receita
df_financeiro_limpo = df_financeiro

for coluna in ["budget", "revenue"]:
    texto = regexp_replace(
        upper(trim(col(coluna).cast("string"))),
        r"USD|[$,\s]",
        ""
    )

    multiplicador = (
        when(texto.rlike("K$"), lit(1000.0))
        .when(texto.rlike("M$"), lit(1000000.0))
        .when(texto.rlike("B$"), lit(1000000000.0))
        .otherwise(lit(1.0))
    )

    # Remover o sufixo K, M ou B
    numero = regexp_replace(texto, r"[KMB]$", "")

    # Converter textos inválidos em NULL com segurança
    valor_numerico = expr(
        f"""try_cast(
            CASE
                WHEN upper(trim(cast(`{coluna}` AS STRING)))
                     IN ('UNKNOWN', 'NÃO INFORMADO', 'N/A', '')
                THEN NULL
                ELSE regexp_replace(
                    regexp_replace(
                        upper(trim(cast(`{coluna}` AS STRING))),
                        '(?i)USD|\\\\$|,|\\\\s',
                        ''
                    ),
                    '[KMB]$',
                    ''
                )
            END AS DECIMAL(18,2)
        )"""
    )

    df_financeiro_limpo = (
        df_financeiro_limpo
        .withColumn(
            coluna + "_usd",
            (valor_numerico * multiplicador).cast("decimal(18,2)")
        )
        .withColumn(
            coluna + "_usd",
            when(
                col(coluna + "_usd") > 0,
                col(coluna + "_usd")
            )
        )
    )

df_financeiro_limpo = df_financeiro_limpo.select(
    col("id"),
    col("budget_usd").alias("orcamento_usd"),
    col("revenue_usd").alias("receita_usd"),
    col("ingestion_datetime")
)

display(df_financeiro_limpo.limit(10))


# 3. Buscar a cotação mais recente disponível
cotacao_dolar = (
    spark.table("bronze.tb_cotacao_dolar")
    .orderBy(col("dataHoraCotacao").desc())
    .select(col("cotacaoCompra").cast("double").alias("cotacaoCompra"))
    .first()["cotacaoCompra"]
)

if cotacao_dolar is None:
    raise ValueError("Não foi encontrada uma cotação válida do dólar.")

print(f"Cotação utilizada: R$ {cotacao_dolar:.4f}")

# 4. Calcular lucro, margem e conversões para reais
df_financeiro_final = (
    df_financeiro_limpo
    .withColumn(
        "lucro_usd",
        (col("receita_usd") - col("orcamento_usd")).cast("decimal(18,2)")
    )
    .withColumn(
        "margem_lucro",
        when(
            col("receita_usd").isNotNull()
            & (col("receita_usd") > 0)
            & col("orcamento_usd").isNotNull(),
            round(
                (col("receita_usd") - col("orcamento_usd"))
                / col("receita_usd") * 100,
                2
            )
        )
    )
    .withColumn(
        "orcamento_brl",
        (col("orcamento_usd") * lit(cotacao_dolar)).cast("decimal(18,2)")
    )
    .withColumn(
        "receita_brl",
        (col("receita_usd") * lit(cotacao_dolar)).cast("decimal(18,2)")
    )
    .withColumn(
        "lucro_brl",
        (col("receita_brl") - col("orcamento_brl")).cast("decimal(18,2)")
    )
    .select(
        col("id").cast("string").alias("id_filme"),
        "orcamento_usd",
        "receita_usd",
        "lucro_usd",
        "margem_lucro",
        "orcamento_brl",
        "receita_brl",
        "lucro_brl",
        "ingestion_datetime"
    )
)

# 5. Conferir os dados antes de salvar
display(df_financeiro_final.limit(10))

# 6. Salvar a tabela Silver
df_financeiro_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.tb_financeiro_filmes")

display(spark.table("silver.tb_financeiro_filmes").limit(10))


In [0]:
%sql
SELECT
    id_filme,
    orcamento_usd,
    receita_usd,
    lucro_usd,
    margem_lucro,
    orcamento_brl,
    receita_brl,
    lucro_brl
FROM silver.tb_financeiro_filmes
LIMIT 10;

In [0]:
df_metricas = spark.table("bronze.tb_movies_metrics")

display(df_metricas)

In [0]:

from pyspark.sql.functions import col, expr, when, regexp_replace

df_metricas_limpa = (
    df_metricas

    # Popularidade: converter vírgula decimal e limpar valores inválidos
    .withColumn(
        "popularidade",
        expr("""
            try_cast(
                regexp_replace(trim(popularity), ',', '.')
                AS DOUBLE
            )
        """)
    )
    .withColumn(
        "popularidade",
        when(col("popularidade") >= 0, col("popularidade"))
    )

    # Média de votos do TMDB: escala de 0 a 10
    .withColumn(
        "media_votos",
        expr("try_cast(vote_average AS DOUBLE)")
    )
    .withColumn(
        "media_votos",
        when(
            col("media_votos").between(0, 10),
            col("media_votos")
        )
    )

    # Quantidade de votos do TMDB
    .withColumn(
        "quantidade_votos",
        expr("try_cast(vote_count AS INT)")
    )
    .withColumn(
        "quantidade_votos",
        when(col("quantidade_votos") >= 0, col("quantidade_votos"))
    )

    # Nota média do IMDb: escala de 0 a 10
    .withColumn(
        "avaliacao_media",
        expr("try_cast(averageRating AS DOUBLE)")
    )
    .withColumn(
        "avaliacao_media",
        when(
            col("avaliacao_media").between(0, 10),
            col("avaliacao_media")
        )
    )

    # Número de votos do IMDb
    .withColumn(
        "numero_votos",
        expr("try_cast(numVotes AS BIGINT)")
    )
    .withColumn(
        "numero_votos",
        when(col("numero_votos") >= 0, col("numero_votos"))
    )
)

display(
    df_metricas_limpa.select(
        "id",
        "popularidade",
        "media_votos",
        "quantidade_votos",
        "avaliacao_media",
        "numero_votos"
    )
)


In [0]:
df_avaliacoes = spark.table("bronze.tb_movies_reviews")

df_avaliacoes_limpa = (
    df_avaliacoes
    .dropDuplicates()
    .withColumn(
        "avaliacao",
        expr("""
            CASE
                WHEN try_cast(nota AS DOUBLE) BETWEEN 0 AND 10
                THEN try_cast(nota AS DOUBLE)
                ELSE NULL
            END
        """)
    )
    .withColumn(
        "comentario_tratado",
        when(
            col("comentario").isNull() | (trim(col("comentario")) == ""),
            "Sem comentário"
        ).otherwise(col("comentario"))
    )
)

display(df_avaliacoes_limpa)

In [0]:

df_avaliacoes_final = df_avaliacoes_limpa.select(
    col("id").cast("string").alias("id_filme"),
    col("nome").alias("nome_usuario"),
    col("avaliacao"),
    col("comentario_tratado").alias("comentario"),
    col("ingestion_datetime")
)

display(df_avaliacoes_final)

df_avaliacoes_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.tb_avaliacoes_usuarios")

display(spark.table("silver.tb_avaliacoes_usuarios"))


In [0]:

df_metricas_final = df_metricas_limpa.select(
    col("id").cast("string").alias("id_filme"),
    col("popularidade"),
    col("media_votos"),
    col("quantidade_votos"),
    col("avaliacao_media"),
    col("numero_votos"),
    col("ingestion_datetime")
)


In [0]:

from pyspark.sql.functions import col

# Padronizar o tipo da coluna de contagem de votos
df_metricas_final = df_metricas_final.withColumn(
    "numero_votos",
    col("numero_votos").cast("long")
)

# Sobrescrever a tabela com o esquema atualizado
df_metricas_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.tb_metricas_engajamento")

display(
    spark.table("silver.tb_metricas_engajamento").limit(10)
)


In [0]:
display(spark.table("silver.tb_metricas_engajamento"))

In [0]:

from pyspark.sql.functions import (
    col, explode, split, trim, initcap, regexp_replace
)

df_generos = spark.table("bronze.tb_credits_and_tags")

df_generos_limpo = (
    df_generos
    .select(
        col("id").cast("string").alias("id_filme"),
        explode(
            split(
                regexp_replace(col("genres"), ";", ","),
                ","
            )
        ).alias("genero")
    )
    .withColumn("genero", initcap(trim(col("genero"))))
    .filter(
        col("genero").isNotNull()
        & (col("genero") != "")
        & ~col("genero").rlike("^[0-9]+$")
    )
    .dropDuplicates(["id_filme", "genero"])
)

display(df_generos_limpo)


In [0]:

from pyspark.sql.functions import col

df_generos_final = (
    df_generos_limpo
    .withColumn("id_filme", col("id_filme").cast("string"))
    .withColumn("genero", col("genero").cast("string"))
    .dropDuplicates(["id_filme", "genero"])
)

df_generos_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.tb_generos")

display(spark.table("silver.tb_generos"))


In [0]:
from pyspark.sql.functions import explode, split, trim, initcap, lit

df_creditos = spark.table("bronze.tb_credits_and_tags")

df_pessoas = (
    df_creditos
    .select(
        col("id").alias("id_filme"),
        explode(split(col("cast"), ",")).alias("nome"),
        lit("Ator").alias("tipo")
    )
)

df_diretores = (
    df_creditos
    .select(
        col("id").alias("id_filme"),
        explode(split(col("directors"), ",")).alias("nome"),
        lit("Diretor").alias("tipo")
    )
)

df_roteiristas = (
    df_creditos
    .select(
        col("id").alias("id_filme"),
        explode(split(col("writers"), ",")).alias("nome"),
        lit("Roteirista").alias("tipo")
    )
)

df_produtoras = (
    df_creditos
    .select(
        col("id").alias("id_filme"),
        explode(
            split(
                regexp_replace(col("production_companies"), ";", ","),
                ","
            )
        ).alias("nome"),
        lit("Produtora").alias("tipo")
    )
)

df_pessoas_empresas = (
    df_pessoas
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_produtoras)
    .withColumn("nome", initcap(trim(col("nome"))))
    .filter(col("nome").isNotNull() & (col("nome") != ""))
    .dropDuplicates(["id_filme", "nome", "tipo"])
)

display(df_pessoas_empresas)

In [0]:
df_pessoas_empresas_final = (
    df_pessoas_empresas
    .withColumn("id_filme", col("id_filme").cast("string"))
    .withColumn("nome", col("nome").cast("string"))
    .withColumn("tipo", col("tipo").cast("string"))
    .dropDuplicates(["id_filme", "nome", "tipo"])
)

df_pessoas_empresas_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.tb_pessoas_empresas")

display(spark.table("silver.tb_pessoas_empresas"))

In [0]:

from pyspark.sql.functions import (
    col, to_date, to_timestamp, sequence, explode,
    min as spark_min, max as spark_max, last, row_number
)
from pyspark.sql.window import Window

# Ler as cotações da Bronze
df_cotacao_base = (
    spark.table("bronze.tb_cotacao_dolar")
    .withColumn(
        "data_hora_cotacao",
        to_timestamp(col("dataHoraCotacao"))
    )
    .withColumn(
        "data_cotacao",
        to_date(col("data_hora_cotacao"))
    )
    .filter(
        col("data_cotacao").isNotNull()
        & col("cotacaoCompra").isNotNull()
    )
)

# Selecionar a cotação mais recente de cada dia
janela_dia = Window.partitionBy("data_cotacao").orderBy(
    col("data_hora_cotacao").desc()
)

df_cotacao_base = (
    df_cotacao_base
    .withColumn("rn", row_number().over(janela_dia))
    .filter(col("rn") == 1)
    .select("data_cotacao", "cotacaoCompra")
)

# Encontrar o período disponível
limites = df_cotacao_base.agg(
    spark_min("data_cotacao").alias("data_inicio"),
    spark_max("data_cotacao").alias("data_fim")
).first()

# Criar calendário diário
df_calendario = spark.createDataFrame(
    [(limites["data_inicio"], limites["data_fim"])],
    ["data_inicio", "data_fim"]
).select(
    explode(
        sequence(col("data_inicio"), col("data_fim"))
    ).alias("data_cotacao")
)

# Preencher dias sem cotação com a última cotação conhecida
janela = (
    Window.orderBy("data_cotacao")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_cotacao_final = (
    df_calendario
    .join(df_cotacao_base, on="data_cotacao", how="left")
    .withColumn(
        "cotacaoCompra",
        last("cotacaoCompra", ignorenulls=True).over(janela)
    )
    .select(
        col("data_cotacao"),
        col("cotacaoCompra").cast("decimal(10,4)")
    )
)

display(df_cotacao_final.orderBy("data_cotacao"))


In [0]:
df_cotacao_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.tb_cotacao_dolar")

display(
    spark.table("silver.tb_cotacao_dolar")
    .orderBy("data_cotacao")
)

In [0]:

%sql
SELECT
    id_filme,
    orcamento_usd,
    orcamento_brl,
    receita_usd,
    receita_brl
FROM silver.tb_financeiro_filmes
WHERE orcamento_usd IS NOT NULL
   OR receita_usd IS NOT NULL
LIMIT 10;


In [0]:

tabelas_silver = [
    "tb_info_filmes",
    "tb_financeiro_filmes",
    "tb_metricas_engajamento",
    "tb_avaliacoes_usuarios",
    "tb_generos",
    "tb_pessoas_empresas",
    "tb_cotacao_dolar"
]

for tabela in tabelas_silver:
    print(f"\nTabela: silver.{tabela}")
    if spark.catalog.tableExists(f"silver.{tabela}"):
        print("Status: OK")
        print("Quantidade de registros:", spark.table(f"silver.{tabela}").count())
    else:
        print("Status: NÃO ENCONTRADA")


In [0]:
%sql
SHOW TABLES IN silver;

In [0]:
tabelas_silver = [
    "tb_info_filmes",
    "tb_financeiro_filmes",
    "tb_metricas_engajamento",
    "tb_avaliacoes_usuarios",
    "tb_generos",
    "tb_pessoas_empresas",
    "tb_cotacao_dolar"
]

for tabela in tabelas_silver:
    print(f"\nTabela: silver.{tabela}")

    if spark.catalog.tableExists(f"silver.{tabela}"):
        print("Status: OK")
        print(
            "Quantidade de registros:",
            spark.table(f"silver.{tabela}").count()
        )
    else:
        print("Status: NÃO ENCONTRADA")